In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName("RDD_Operations").getOrCreate()

In [3]:
customer_data = [
    "customer_id,name,city,state,country,registration_date,is_active",
    "0,Customer_0,Bangalore,Karnataka,India,2023-11-11,True",
    "1,Customer_1,Hyderabad,Delhi,India,2023-08-26,True",
    "2,Customer_2,Ahmedabad,West Bengal,India,2023-06-23,True",
    "3,Customer_3,Bangalore,Tamil Nadu,India,2023-03-24,False",
    "4,Customer_4,Bangalore,Gujarat,India,2023-06-06,False",
    "5,Customer_5,Delhi,Maharashtra,India,2023-04-19,False"
]

In [4]:
data_rdd = spark.sparkContext.parallelize(customer_data)

In [6]:
data_rdd.getNumPartitions()

2

In [7]:
# RDD - Resilient Distributed Dataset

# first() - returns the first element of RDD

In [9]:
header = data_rdd.first()

In [10]:
header

'customer_id,name,city,state,country,registration_date,is_active'

In [11]:
data_rdd = data_rdd.filter(lambda row : row!=header)

In [12]:
data_rdd

PythonRDD[2] at RDD at PythonRDD.scala:57

## Map() - It applied a fucntion to each element in RDD

In [13]:
data_rdd.collect()

['0,Customer_0,Bangalore,Karnataka,India,2023-11-11,True',
 '1,Customer_1,Hyderabad,Delhi,India,2023-08-26,True',
 '2,Customer_2,Ahmedabad,West Bengal,India,2023-06-23,True',
 '3,Customer_3,Bangalore,Tamil Nadu,India,2023-03-24,False',
 '4,Customer_4,Bangalore,Gujarat,India,2023-06-06,False',
 '5,Customer_5,Delhi,Maharashtra,India,2023-04-19,False']

In [23]:
def parse_row(row):
  fields = row.split(',')
  return (
      int (fields[0]),
      fields[1],
      fields[2],
      fields[3],
      fields[4],
      fields[5],
      fields[6] == 'True'
  )



In [24]:
parsed_rdd = data_rdd.map(parse_row)

In [25]:
parsed_rdd.collect()

[(0, 'Customer_0', 'Bangalore', 'Karnataka', 'India', '2023-11-11', True),
 (1, 'Customer_1', 'Hyderabad', 'Delhi', 'India', '2023-08-26', True),
 (2, 'Customer_2', 'Ahmedabad', 'West Bengal', 'India', '2023-06-23', True),
 (3, 'Customer_3', 'Bangalore', 'Tamil Nadu', 'India', '2023-03-24', False),
 (4, 'Customer_4', 'Bangalore', 'Gujarat', 'India', '2023-06-06', False),
 (5, 'Customer_5', 'Delhi', 'Maharashtra', 'India', '2023-04-19', False)]

## Advanced RDD Operations
*italicised text*

### Extract a field with Map() - customer and city

In [29]:
name_city_rdd = parsed_rdd.map(lambda row : (row[1], row[2]))

In [30]:
name_city_rdd

PythonRDD[8] at RDD at PythonRDD.scala:57

In [31]:
name_city_rdd.first()

('Customer_0', 'Bangalore')

In [33]:
active_customers = parsed_rdd.filter(lambda row:row[6]== True)
active_customers.collect()

[(0, 'Customer_0', 'Bangalore', 'Karnataka', 'India', '2023-11-11', True),
 (1, 'Customer_1', 'Hyderabad', 'Delhi', 'India', '2023-08-26', True),
 (2, 'Customer_2', 'Ahmedabad', 'West Bengal', 'India', '2023-06-23', True)]

## distinct () - Transformation

In [34]:
cities_rdd = parsed_rdd.map(lambda row:row[2]).distinct()

In [35]:
cities_rdd.collect()

['Hyderabad', 'Delhi', 'Bangalore', 'Ahmedabad']

In [36]:
cities_rdd.take(1)

['Hyderabad']

### Reduce by key transformation

In [37]:
## combines the values for each keu using an associative reduce function .

In [38]:
customer_per_city = parsed_rdd.map(lambda row:(row[2],1)).reduceByKey(lambda x,y:x+y)

In [40]:
customer_per_city.collect()

[('Hyderabad', 1), ('Delhi', 1), ('Bangalore', 3), ('Ahmedabad', 1)]

In [41]:
# CountByValue

In [42]:
customer_per_city = parsed_rdd.map(lambda row:row[2].countByValues())

In [43]:
customer_per_city

PythonRDD[22] at RDD at PythonRDD.scala:57

# imp - ReduceByKey is a Transformation while CountByValues is an Action

## Combine more operations

In [44]:
# cities with active customer

In [48]:
active_cities = parsed_rdd \
    .filter(lambda row: row[6]) \
    .map(lambda row: row[2]) \
    .distinct()

In [50]:
active_cities.collect()

['Hyderabad', 'Bangalore', 'Ahmedabad']